In [1]:
import os
import ray
import json
import pickle
from dynaconf import Dynaconf
from tqdm.notebook import tqdm
from utils import check_path, convert_np_arrays, flatten_dict, env_creator
from ray.tune.logger import JsonLogger
from algorithms_with_statistics.ddqn_pber import DDQNWithMPBERAndLogging
from replay_buffer.mpber_ram_saver import MultiAgentPrioritizedBlockReplayBuffer
from ray.tune.registry import register_env

pygame 2.5.2 (SDL 2.28.2, Python 3.9.16)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# Init Ray
ray.init(
    num_cpus=5, num_gpus=1,
    include_dashboard=False,
    _system_config={"maximum_gcs_destroyed_actor_cached_count": 20},
)

2024-04-06 14:52:55,189	INFO worker.py:1673 -- Started a local Ray instance.


Python version:,3.9.16
Ray version:,2.8.0


In [3]:
import mlflow
from mlflow.exceptions import MlflowException
from func_timeout import FunctionTimedOut
from botocore.exceptions import ConnectionClosedError

In [4]:
import datetime
# Config path
env_name = "Breakout"
run_name = int(datetime.datetime.now().timestamp())
log_path = "/home/seventheli/data/BER/experiments/logging/%s" % env_name
checkpoint_path = "/home/seventheli/data/BER/experiments/checkpoints/%s" % env_name
# Set mlflow
mlflow.set_tracking_uri("http://localhost:9999")
mlflow.set_experiment(experiment_name=env_name)
mlflow_client = mlflow.tracking.MlflowClient()

In [5]:
# Check path available
import shutil
check_path(log_path)
log_path = str(os.path.join(log_path, str(run_name)))
check_path(log_path)
if os.path.exists(log_path):
    shutil.rmtree(log_path)
check_path(log_path)
check_path(checkpoint_path)
checkpoint_path = os.path.join(checkpoint_path, str(run_name))
check_path(checkpoint_path)
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
check_path(checkpoint_path)

In [6]:
# Set hyper parameters
setting = "./settings/ddqn_atari.yml"
setting = Dynaconf(envvar_prefix="DYNACONF", settings_files=setting)

hyper_parameters = setting.hyper_parameters.to_dict()
hyper_parameters["logger_config"] = {"type": JsonLogger, "logdir": checkpoint_path}
hyper_parameters["env_config"] = {
    "id": env_name + "NoFrameskip-v4",
}

In [7]:
# Set run object
run_name = "DDQN_%s" % env_name + "_PBER_RAM_SAVER_%s" % run_name
env_example = env_creator(hyper_parameters["env_config"])
obs, _ = env_example.reset()
step = env_example.step(1)
print(env_example.action_space, env_example.observation_space)
print(env_example)
print("log path: %s; check_path: %s" % (log_path, checkpoint_path))
register_env("Atari", env_creator)

Discrete(4) Box(0, 255, (84, 84, 4), uint8)
<FrameStack<WarpFrame<FireResetEnv<EpisodicLifeEnv<MaxAndSkipEnv<NoopResetEnv<MonitorEnv<OrderEnforcing<PassiveEnvChecker<AtariEnv<BreakoutNoFrameskip-v4>>>>>>>>>>>
log path: /home/seventheli/data/BER/experiments/logging/Breakout/1712411576; check_path: /home/seventheli/data/BER/experiments/checkpoints/Breakout/1712411576


A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


In [8]:
# Set Replay Buffer
replay_buffer_config = {
    **hyper_parameters["replay_buffer_config"],
    "type": MultiAgentPrioritizedBlockReplayBuffer,
    "obs_space": env_example.observation_space,
    "action_space": env_example.action_space,
    "sub_buffer_size": 8,
    "worker_side_prioritization": False,
    "replay_buffer_shards_colocated_with_driver": True,
    "rollout_fragment_length": hyper_parameters["rollout_fragment_length"],
    "num_save": 50,
    "split_mini_batch": 1,
    "store": 3000
}
hyper_parameters["replay_buffer_config"] = replay_buffer_config
hyper_parameters["optimizer"] = {"num_replay_buffer_shards": 1}

In [9]:
# Set Trainer
trainer = DDQNWithMPBERAndLogging(config=hyper_parameters, env="Atari")

2024-04-06 14:52:56,795	WARNING deprecation.py:50 -- DeprecationWarning: `rllib/algorithms/simple_q/` has been deprecated. Use `rllib_contrib/simple_q/` instead. This will raise an error in the future!
2024-04-06 14:52:56,797	WARNING deprecation.py:50 -- DeprecationWarning: `algo = Algorithm(env='Atari', ...)` has been deprecated. Use `algo = AlgorithmConfig().environment('Atari').build()` instead. This will raise an error in the future!
/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/ray/rllib/utils/from_config.py:197: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  object_ = constructor(*ctor_args, **ctor_kwargs)
2024-04-06 14:52:56,995	WARNING deprecation.py:50 -- DeprecationWarning: `ray.

In [10]:
# Common setup
checkpoint_path = str(checkpoint_path)
# Save initial configuration
with open(os.path.join(checkpoint_path, "%s_config.pyl" % run_name), "wb") as f:
    _ = trainer.config.to_dict()
    pickle.dump(_, f)

In [11]:
mlflow_run = mlflow.start_run(run_name=run_name,
                              tags={"mlflow.user": "10900+3090"})
# Log parameters
mlflow.log_params(hyper_parameters["replay_buffer_config"])
to_log = ['double_q', 'dueling', 'lr', 'n_step', 'num_steps_sampled_before_learning_starts',
          'rollout_fragment_length', 'target_network_update_freq', 'train_batch_size', 'min_sample_timesteps_per_iteration']
mlflow.log_params(
    {key: hyper_parameters[key] for key in to_log})

In [12]:
keys_to_extract_sam = {"episode_reward_max", "episode_reward_min", "episode_reward_mean"}
keys_to_extract_sta = {"num_agent_steps_sampled", "num_agent_steps_trained"}
keys_to_extract_buf = {"add_batch_time_ms", "replay_time_ms", "update_priorities_time_ms"}

In [13]:
mlflow.pytorch.log_model(trainer.get_policy().model, run_name)
model_uri = "runs:/%s/model_name" % mlflow_run.info.run_id
mlflow.register_model(model_uri, run_name, tags={"episode" : 0})

/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
Successfully registered model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576'.
2024/04/06 14:53:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_RAM_SAVER_1712411576, version 1
Created version '1' of model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576'.


<ModelVersion: aliases=[], creation_timestamp=1712411586902, current_stage='None', description='', last_updated_timestamp=1712411586902, name='DDQN_Breakout_PBER_RAM_SAVER_1712411576', run_id='d937e3b90dcc40339639a83c55092400', run_link='', source='s3://jo-mlflow-ber/1/d937e3b90dcc40339639a83c55092400/artifacts/model_name', status='READY', status_message='', tags={'episode': '0'}, user_id='', version='1'>

In [14]:
for i in tqdm(range(0, setting.log.max_run), ascii=True):
    result = trainer.train()
    time_used = result["time_total_s"]
    try:
        sampler = result.get("sampler_results", {}).copy()
        eva = result.get("evaluation", {}).copy()
        info = result.get("info", {}).copy()
        sam = {key: sampler[key] for key in keys_to_extract_sam if key in sampler}
        sta = {key: info[key] for key in keys_to_extract_sta if key in info}
        buf = flatten_dict(trainer.local_replay_buffer.stats())
        buf["est_size_gb"] = buf["est_size_bytes"] /1e9
        result['buffer'] = buf
        time_usage = info.get("learner", {}).get("time_usage", {})
        if eva:
            eva = {"eval_" + key: eva[key] for key in keys_to_extract_sam if key in eva}
        mlflow.log_metrics({**sam, **sta, **buf, **time_usage, **eva}, step=result["episodes_total"])
        if i % (setting.log.log * 50) == 0:
            trainer.save_checkpoint(checkpoint_path)
            mlflow.pytorch.log_model(trainer.get_policy().model, run_name)
            mlflow.register_model(model_uri, run_name, tags={
                "episode" : result["episodes_total"],
                "reward": sam["episode_reward_mean"],
            })
            mlflow.log_artifacts(log_path)
            mlflow.log_artifacts(checkpoint_path)
        if i % 10 == 0:
            tqdm.write("episode %d ; " % result["episodes_total"] +  " ".join(["%s : %f8" % (i, j)for i, j in sam.items()]))
    except FunctionTimedOut:
        tqdm.write("logging failed")
    except MlflowException:
        tqdm.write("logging failed")
    except ConnectionClosedError:
        tqdm.write("logging failed")
    with open(os.path.join(log_path, str(i) + ".json"), "w") as f:
        result["config"] = None
        json.dump(convert_np_arrays(result), f)
    if time_used >= setting.log.max_time:
        break

  0%|          | 0/20000 [00:00<?, ?it/s]

2024-04-06 14:53:07,248	WARNING multi_agent_prioritized_replay_buffer.py:215 -- Adding batches with column `weights` to this buffer while providing weights as a call argument to the add method results in the column being overwritten.
Registered model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576' already exists. Creating a new version of this model...
2024/04/06 14:54:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_RAM_SAVER_1712411576, version 2
Created version '2' of model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576'.


episode 64 ; episode_reward_mean : 1.0156258 episode_reward_min : 0.0000008 episode_reward_max : 4.0000008


2024-04-06 15:00:23,785	WARNING deprecation.py:50 -- DeprecationWarning: `ray.rllib.execution.train_ops.multi_gpu_train_one_step` has been deprecated. This will raise an error in the future!


episode 675 ; episode_reward_mean : 0.8900008 episode_reward_min : 0.0000008 episode_reward_max : 6.0000008
episode 1380 ; episode_reward_mean : 0.8100008 episode_reward_min : 0.0000008 episode_reward_max : 4.0000008
episode 1778 ; episode_reward_mean : 10.1100008 episode_reward_min : 0.0000008 episode_reward_max : 36.0000008
episode 1985 ; episode_reward_mean : 13.6400008 episode_reward_min : 7.0000008 episode_reward_max : 29.0000008
episode 2152 ; episode_reward_mean : 16.0800008 episode_reward_min : 7.0000008 episode_reward_max : 33.0000008
episode 2301 ; episode_reward_mean : 26.3400008 episode_reward_min : 10.0000008 episode_reward_max : 234.0000008
episode 2437 ; episode_reward_mean : 30.0400008 episode_reward_min : 11.0000008 episode_reward_max : 237.0000008
episode 2561 ; episode_reward_mean : 31.6200008 episode_reward_min : 17.0000008 episode_reward_max : 75.0000008
episode 2674 ; episode_reward_mean : 37.2400008 episode_reward_min : 13.0000008 episode_reward_max : 265.0000008

...

episode 10507 ; episode_reward_mean : 176.3200008 episode_reward_min : 13.0000008 episode_reward_max : 382.0000008
episode 10590 ; episode_reward_mean : 191.7700008 episode_reward_min : 7.0000008 episode_reward_max : 407.0000008
episode 10669 ; episode_reward_mean : 170.2500008 episode_reward_min : 7.0000008 episode_reward_max : 407.0000008
episode 10749 ; episode_reward_mean : 188.6600008 episode_reward_min : 6.0000008 episode_reward_max : 401.0000008
episode 10831 ; episode_reward_mean : 163.5400008 episode_reward_min : 6.0000008 episode_reward_max : 386.0000008
episode 10913 ; episode_reward_mean : 166.2000008 episode_reward_min : 15.0000008 episode_reward_max : 387.0000008
episode 11007 ; episode_reward_mean : 134.2100008 episode_reward_min : 9.0000008 episode_reward_max : 407.0000008
episode 11103 ; episode_reward_mean : 122.4200008 episode_reward_min : 11.0000008 episode_reward_max : 399.0000008
episode 11190 ; episode_reward_mean : 146.0000008 episode_reward_min : 8.0000008 epis

Registered model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576' already exists. Creating a new version of this model...
2024/04/09 06:41:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_RAM_SAVER_1712411576, version 5
Created version '5' of model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576'.


episode 14797 ; episode_reward_mean : 159.5300008 episode_reward_min : 15.0000008 episode_reward_max : 413.0000008
episode 14883 ; episode_reward_mean : 150.8900008 episode_reward_min : 5.0000008 episode_reward_max : 405.0000008
episode 14969 ; episode_reward_mean : 155.0000008 episode_reward_min : 8.0000008 episode_reward_max : 404.0000008
episode 15053 ; episode_reward_mean : 151.0100008 episode_reward_min : 8.0000008 episode_reward_max : 393.0000008
episode 15136 ; episode_reward_mean : 140.7000008 episode_reward_min : 12.0000008 episode_reward_max : 419.0000008
episode 15219 ; episode_reward_mean : 180.6800008 episode_reward_min : 14.0000008 episode_reward_max : 414.0000008
episode 15305 ; episode_reward_mean : 179.6600008 episode_reward_min : 18.0000008 episode_reward_max : 396.0000008
episode 15393 ; episode_reward_mean : 158.3800008 episode_reward_min : 14.0000008 episode_reward_max : 422.0000008
episode 15475 ; episode_reward_mean : 176.8300008 episode_reward_min : 16.0000008 e

Registered model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576' already exists. Creating a new version of this model...
2024/04/10 03:58:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_RAM_SAVER_1712411576, version 6
Created version '6' of model 'DDQN_Breakout_PBER_RAM_SAVER_1712411576'.


episode 19160 ; episode_reward_mean : 158.1600008 episode_reward_min : 16.0000008 episode_reward_max : 408.0000008
episode 19256 ; episode_reward_mean : 144.2400008 episode_reward_min : 1.0000008 episode_reward_max : 390.0000008
episode 19346 ; episode_reward_mean : 151.9900008 episode_reward_min : 3.0000008 episode_reward_max : 396.0000008
episode 19432 ; episode_reward_mean : 135.5600008 episode_reward_min : 12.0000008 episode_reward_max : 390.0000008
episode 19516 ; episode_reward_mean : 173.3600008 episode_reward_min : 8.0000008 episode_reward_max : 390.0000008
episode 19598 ; episode_reward_mean : 172.8800008 episode_reward_min : 12.0000008 episode_reward_max : 408.0000008
episode 19682 ; episode_reward_mean : 152.1600008 episode_reward_min : 11.0000008 episode_reward_max : 396.0000008
episode 19775 ; episode_reward_mean : 137.0300008 episode_reward_min : 4.0000008 episode_reward_max : 399.0000008
episode 19872 ; episode_reward_mean : 108.9700008 episode_reward_min : 11.0000008 ep

In [ ]:
mlflow.log_artifacts(log_path)
mlflow.log_artifacts(checkpoint_path)

In [ ]:
mlflow.end_run()